# Run the baselines on a GPU machine (Colab or any CUDA box)

1. Push this repo first (`git push origin main` on the Mac).
2. Runtime → GPU (an L4/A100 is comfortable; a T4 works but `make baseline-b` and `make pilot` take several hours).
3. Run the cells top to bottom. Each `make` target is resumable or cheap to re-run; outputs land in `results/`.
4. Download `results.zip` at the end and unzip it into the repo on the Mac (or commit from here).

In [ ]:
!nvidia-smi
!git clone https://github.com/mehul24d/Reasoning-Failure-Diagnosis.git
%cd Reasoning-Failure-Diagnosis
!git log --oneline | head -3

In [ ]:
# Keep Colab's CUDA torch build; install the rest at the pinned versions.
!grep -v '^torch==' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt && pip install -q -e . --no-deps
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Optional: larger batches on a big GPU (edit, don't commit unless you mean it)
import yaml, pathlib
p = pathlib.Path('configs/harness.yaml'); cfg = yaml.safe_load(p.read_text())
cfg['generation']['batch_size'] = 64; cfg['forward_batch_size'] = 16
p.write_text(yaml.safe_dump(cfg, sort_keys=False)); print(cfg)

In [ ]:
!make test-fast && make phase0 && cat results/phase0/equivalence.json | head -40

In [ ]:
!make baseline-a

In [ ]:
!make traces

In [ ]:
!make baseline-b

In [ ]:
!make pilot && make pilot-sample

In [ ]:
!make report && zip -qr results.zip results docs/BASELINE_REPORT_TABLES.md -x '*.npy' '*.npz' && ls -lh results.zip